In [1]:
# === Installation des dépendances ===
!pip install flask flask-cors pillow kaggle -q
!pip install pyngrok -q
# === Importations ===
import os
import re
import shutil
import json
import logging
import threading
import tempfile
import io
import base64
from datetime import datetime
from pathlib import Path
from functools import wraps

from flask import Flask, jsonify, request, send_from_directory
from flask_cors import CORS
from PIL import Image, UnidentifiedImageError

print("✅ Toutes les dépendances installées")

✅ Toutes les dépendances installées


In [ ]:
# =====================================================================
#  SmartBasket Admin — Flask Backend  (v5-fixed4)
#
#  FIX vs v5-fixed3 :a
#    kaggle_push_product() ne copiait QUE le nouveau produit dans push_dir
#    → Kaggle remplaçait tout le dataset par un seul dossier produit.
#
#    SOLUTION :
#      - kaggle_push_product() copie maintenant TOUTE la galerie locale
#        dans push_dir, puis ajoute/écrase le dossier du produit concerné.
#      - Résultat : Kaggle reçoit dataset complet = tous les produits.
# =====================================================================

import os, re, shutil, json, logging, threading, tempfile, io, base64
import pickle, subprocess, sys
from datetime import datetime
from pathlib import Path

from flask import Flask, jsonify, request, Response
from flask_cors import CORS
from PIL import Image

print("✅ Imports OK")

# ── Configuration ────────────────────────────────────────────────────
KAGGLE_IMAGES_DATASET = os.environ.get(
    "KAGGLE_IMAGES_DATASET", "sarahlaouedj25/smartbasket-dataset-multiside"
)
KAGGLE_MODEL_DATASET = os.environ.get(
    "KAGGLE_MODEL_DATASET", "abdelghaniyacine/gallary-dino-76"
)
SERVER_V1_URL           = os.environ.get("SERVER_V1_URL", "https://underfed-rifling-mummify.ngrok-free.dev")           # paste server-v1 ngrok URL here
SERVER_V1_RELOAD_SECRET = os.environ.get("SERVER_V1_RELOAD_SECRET", "smartbasket-reload-secret")
LOCAL_CACHE_DIR = Path(os.environ.get(
    "LOCAL_CACHE_DIR", str(Path.home() / ".smartbasket" / "gallery")
))

YOLO_MODEL_PATH      = os.environ.get(
    "YOLO_MODEL_PATH",
    "/kaggle/input/datasets/sarahlaouedj25/yoloresultobbv5/best(2).pt"
)
FINETUNED_CHECKPOINT = os.environ.get(
    "FINETUNED_CHECKPOINT",
    "/kaggle/input/datasets/abdelghaniyacine/gallary-dino-76/dinov2_finetuned_supcon_v3.pt"
)
GALLERY_PKL_PATH = os.environ.get(
    "GALLERY_PKL_PATH",
    "/kaggle/input/datasets/abdelghaniyacine/gallary-dino-76/gallery_finetuned_v3.pkl"
)
FAISS_INDEX_PATH = os.environ.get(
    "FAISS_INDEX_PATH",
    "/kaggle/input/datasets/abdelghaniyacine/gallary-dino-76/gallery_finetuned_v3.index"
)

# Copies de travail writable
_KAGGLE_WORKING = Path("/kaggle/working")
if _KAGGLE_WORKING.exists() and os.access(str(_KAGGLE_WORKING), os.W_OK):
    _WORK_DIR = _KAGGLE_WORKING / "smartbasket_gallery"
else:
    _WORK_DIR = Path.home() / ".smartbasket"

GALLERY_PKL_WORK = os.environ.get("GALLERY_PKL_WORK",  str(_WORK_DIR / "gallery_finetuned_v3.pkl"))
FAISS_INDEX_WORK = os.environ.get("FAISS_INDEX_WORK",  str(_WORK_DIR / "gallery_finetuned_v3.index"))

CURRENT_DIR  = Path.cwd()
SERVE_STATIC = CURRENT_DIR / "static"
ALLOWED_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff"}
THUMB_SIZE   = (160, 160)
PUSH_NOTES   = "SmartBasket Admin update"
YOLO_DEFAULT_CLASS = 0
YOLO_CONF_THR      = 0.25
OBB_CROP_SIZE      = 336

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("smartbasket")

SERVE_STATIC.mkdir(parents=True, exist_ok=True)
app = Flask(__name__, static_folder=str(SERVE_STATIC))
CORS(app, resources={r"/api/*": {"origins": "*"}}, supports_credentials=False)

print(f"✅ Work dir  : {_WORK_DIR}")
print(f"✅ PKL work  : {GALLERY_PKL_WORK}")
print(f"✅ FAISS work: {FAISS_INDEX_WORK}")


# ════════════════════════════════════════════════════════════════════
#  Modèles (lazy)
# ════════════════════════════════════════════════════════════════════
_models_lock  = threading.Lock()
_models_ready = False
_yolo_model   = None
_dinov2       = None
_dinov2_tfm   = None
_use_ms       = False
_device       = "cpu"

def _ensure_deps():
    for pkg in ["ultralytics", "faiss-cpu", "torch", "torchvision", "timm", "numpy", "Pillow"]:
        try:
            __import__(pkg.split("-")[0])
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

def _load_models():
    global _models_ready, _yolo_model, _dinov2, _dinov2_tfm, _use_ms, _device
    with _models_lock:
        if _models_ready:
            return
        _ensure_deps()
        import torch, timm
        import torch.nn.functional as F
        import torchvision.transforms as T
        from ultralytics import YOLO

        _device = "cuda" if torch.cuda.is_available() else "cpu"
        log.info(f"  Device : {_device}")

        if Path(YOLO_MODEL_PATH).exists():
            try:
                _yolo_model = YOLO(YOLO_MODEL_PATH)
                log.info(f"  YOLO ✓ ({len(_yolo_model.names)} classes)")
            except Exception as e:
                log.warning(f"  YOLO non chargé : {e}")

        def _timm_ns(name):
            orig = torch.nn.Module.load_state_dict
            def _p(self, sd, strict=True, assign=False):
                return orig(self, sd, strict=False, assign=assign)
            torch.nn.Module.load_state_dict = _p
            try:
                return timm.create_model(name, pretrained=True, num_classes=0, global_pool="avg")
            finally:
                torch.nn.Module.load_state_dict = orig

        dino = None
        for fn in [
            lambda: torch.hub.load("facebookresearch/dinov2", "dinov2_vitl14_reg",
                                   pretrained=True, force_reload=False),
            lambda: _timm_ns("vit_large_patch14_reg4_dinov2.lvd142m"),
        ]:
            try:
                dino = fn(); log.info("  DINOv2 ✓"); break
            except Exception as e:
                log.debug(f"  {e}")

        if dino is None:
            log.error("  DINOv2 non disponible")
            _models_ready = True
            return

        dino = dino.to(_device).eval()
        for p in dino.parameters():
            p.requires_grad = False

        if Path(FINETUNED_CHECKPOINT).exists():
            try:
                ckpt = torch.load(FINETUNED_CHECKPOINT, map_location=_device)
                dino.load_state_dict(ckpt["backbone_sd"], strict=False)
                dino.eval()
                log.info("  Fine-tuné ✓")
            except Exception as e:
                log.warning(f"  Fine-tuned non chargé : {e}")

        _dinov2 = dino
        try:
            with open(_gallery_pkl_path(), "rb") as f:
                gdata = pickle.load(f)
            _use_ms = gdata.get("use_multiscale", False)
        except Exception:
            _use_ms = False

        MEAN = [0.485, 0.456, 0.406]; STD = [0.229, 0.224, 0.225]
        _dinov2_tfm = {
            "224": T.Compose([T.Resize((224,224), interpolation=T.InterpolationMode.BICUBIC),
                              T.CenterCrop(224), T.ToTensor(), T.Normalize(MEAN, STD)]),
            "336": T.Compose([T.Resize((336,336), interpolation=T.InterpolationMode.BICUBIC),
                              T.CenterCrop(336), T.ToTensor(), T.Normalize(MEAN, STD)]),
        }
        _models_ready = True


def _gallery_pkl_path():
    return GALLERY_PKL_WORK if Path(GALLERY_PKL_WORK).exists() else GALLERY_PKL_PATH

def _faiss_index_path():
    return FAISS_INDEX_WORK if Path(FAISS_INDEX_WORK).exists() else FAISS_INDEX_PATH


def _ensure_work_copies():
    work_pkl   = Path(GALLERY_PKL_WORK)
    work_faiss = Path(FAISS_INDEX_WORK)
    work_pkl.parent.mkdir(parents=True, exist_ok=True)

    if not work_pkl.exists():
        if Path(GALLERY_PKL_PATH).exists():
            shutil.copy2(GALLERY_PKL_PATH, str(work_pkl))
            log.info(f"  PKL copié ✓ ({work_pkl.stat().st_size/1024/1024:.1f} MB)")
        else:
            import faiss as _faiss
            gdata_empty = {
                "all_labels":[], "labels":[], "embed_dim":1024,
                "use_multiscale":False, "centroids":{}, "visual_signatures":{},
                "global_thr":0.734, "per_label_thr":{}, "label_counts":{},
                "label_to_idxs":{}, "products":[], "n_raw_images":0,
            }
            with open(str(work_pkl), "wb") as f:
                pickle.dump(gdata_empty, f)
            log.info("  PKL vierge créé ✓")

    if not work_faiss.exists():
        if Path(FAISS_INDEX_PATH).exists():
            shutil.copy2(FAISS_INDEX_PATH, str(work_faiss))
            log.info(f"  FAISS copié ✓ ({work_faiss.stat().st_size/1024/1024:.1f} MB)")
        else:
            import faiss as _faiss
            with open(str(work_pkl), "rb") as f:
                _d = pickle.load(f)
            dim = _d.get("embed_dim", 1024)
            _faiss.write_index(_faiss.IndexFlatIP(dim), str(work_faiss))
            log.info(f"  FAISS vierge créé ✓")

    log.info(f"  Work dir : {work_pkl.parent}")
    log.info(f"    PKL   : {work_pkl} ({work_pkl.stat().st_size/1024/1024:.1f} MB)")
    log.info(f"    FAISS : {work_faiss} ({work_faiss.stat().st_size/1024/1024:.1f} MB)")


# ════════════════════════════════════════════════════════════════════
#  YOLO
# ════════════════════════════════════════════════════════════════════
def _run_yolo_on_image(img_path: Path, label: str):
    if _yolo_model is None:
        return 0
    import numpy as np, cv2
    frame = cv2.imread(str(img_path))
    if frame is None:
        return 0
    fh, fw = frame.shape[:2]
    results = _yolo_model(frame, conf=YOLO_CONF_THR, verbose=False)
    if not results or results[0].obb is None or len(results[0].obb) == 0:
        (prod_root(label) / "labels" / (img_path.stem + ".txt")).write_text("")
        return 0
    obb = results[0].obb
    lines = []
    for i, (cx, cy, w, h, angle) in enumerate(obb.xywhr.cpu().numpy()):
        cls = int(obb.cls[i].item()) if hasattr(obb, "cls") else YOLO_DEFAULT_CLASS
        lines.append(f"{cls} {cx/fw:.6f} {cy/fh:.6f} {w/fw:.6f} {h/fh:.6f} {angle:.6f}")
        try:
            import numpy as np
            ca, sa = np.cos(angle), np.sin(angle)
            pw, ph = w*1.07, h*1.07
            c   = np.array([[-pw/2,-ph/2],[pw/2,-ph/2],[pw/2,ph/2],[-pw/2,ph/2]], dtype=np.float32)
            R   = np.array([[ca,-sa],[sa,ca]], dtype=np.float32)
            pts = (c @ R.T) + np.array([cx,cy], dtype=np.float32)
            dst = np.array([[0,0],[OBB_CROP_SIZE,0],[OBB_CROP_SIZE,OBB_CROP_SIZE],[0,OBB_CROP_SIZE]], dtype=np.float32)
            M, _ = cv2.findHomography(pts, dst, method=0)
            if M is not None:
                crop = cv2.warpPerspective(frame, M, (OBB_CROP_SIZE,OBB_CROP_SIZE),
                                           flags=cv2.INTER_LANCZOS4, borderMode=cv2.BORDER_REPLICATE)
                if crop is not None and crop.size > 0:
                    p = prod_root(label) / "croppedimages" / f"{img_path.stem}_crop{i:02d}.jpg"
                    cv2.imwrite(str(p), crop, [cv2.IMWRITE_JPEG_QUALITY, 90])
        except Exception as e:
            log.debug(f"  Crop {i} : {e}")
    (prod_root(label) / "labels" / (img_path.stem + ".txt")).write_text("\n".join(lines))
    return len(lines)


# ════════════════════════════════════════════════════════════════════
#  DINOv2 embeddings
# ════════════════════════════════════════════════════════════════════
def _encode_image(pil_img):
    if _dinov2 is None or _dinov2_tfm is None:
        return None
    import torch, torch.nn.functional as F
    with torch.no_grad():
        t224 = _dinov2_tfm["224"](pil_img.convert("RGB")).unsqueeze(0).to(_device)
        f224 = F.normalize(_dinov2(t224), p=2, dim=1)
        if _use_ms:
            t336 = _dinov2_tfm["336"](pil_img.convert("RGB")).unsqueeze(0).to(_device)
            f336 = F.normalize(_dinov2(t336), p=2, dim=1)
            emb  = F.normalize(torch.cat([f224, f336], dim=1), p=2, dim=1)
        else:
            emb = f224
    return emb.cpu().numpy()[0].astype("float32")


def _add_embeddings_to_gallery(label: str, image_paths: list) -> int:
    if _dinov2 is None:
        log.warning("  DINOv2 non dispo → pas d'embeddings")
        return 0

    import faiss, numpy as np, torch

    _ensure_work_copies()
    work_pkl   = Path(GALLERY_PKL_WORK)
    work_faiss = Path(FAISS_INDEX_WORK)

    with open(str(work_pkl), "rb") as f:
        gdata = pickle.load(f)

    dim          = gdata.get("embed_dim", 1024)
    all_labels   = list(gdata.get("all_labels", gdata.get("labels", [])))
    label_to_idx = dict(gdata.get("label_to_idxs", {}))
    centroids    = dict(gdata.get("centroids", {}))
    label_counts = dict(gdata.get("label_counts", {}))
    products     = list(gdata.get("products", sorted(set(all_labels))))

    index = faiss.read_index(str(work_faiss))
    if index.d != dim:
        raise ValueError(f"FAISS dim ({index.d}) ≠ PKL embed_dim ({dim})")

    log.info(f"  PKL chargé : {len(all_labels)} labels, dim={dim}, {len(products)} produits")
    log.info(f"  FAISS chargé : {index.ntotal} vecteurs")

    new_embs = []
    with torch.no_grad():
        for img_path in image_paths:
            try:
                emb = _encode_image(Image.open(str(img_path)))
                if emb is None or len(emb) != dim:
                    continue
                new_embs.append(emb)
                log.info(f"  Embedding : {img_path.name}")
            except Exception as e:
                log.warning(f"  Embed échoué {img_path.name} : {e}")

    if not new_embs:
        log.warning(f"  Aucun embedding pour '{label}'")
        return 0

    batch     = np.stack(new_embs).astype("float32")
    start_idx = index.ntotal
    index.add(batch)

    for i, lbl in enumerate(new_embs):
        all_labels.append(label)
        label_to_idx.setdefault(label, []).append(start_idx + i)

    idxs = label_to_idx.get(label, [])
    if idxs:
        mat  = np.zeros((len(idxs), dim), dtype="float32")
        for k, gi in enumerate(idxs):
            if gi < index.ntotal:
                mat[k] = index.reconstruct(gi)
        c = mat.mean(axis=0).astype("float32")
        n = np.linalg.norm(c)
        centroids[label]    = c / n if n > 0 else c
        label_counts[label] = len(idxs)

    if label not in products:
        products.append(label)

    gdata.update({
        "all_labels":    all_labels,
        "labels":        all_labels,
        "centroids":     centroids,
        "label_counts":  label_counts,
        "label_to_idxs": label_to_idx,
        "products":      sorted(set(products)),
        "n_raw_images":  index.ntotal,
    })

    # Sauvegarde atomique
    pkl_tmp = work_pkl.with_suffix(".pkl.tmp")
    with open(str(pkl_tmp), "wb") as f:
        pickle.dump(gdata, f)
    pkl_tmp.replace(work_pkl)

    faiss_tmp = work_faiss.with_suffix(".index.tmp")
    faiss.write_index(index, str(faiss_tmp))
    faiss_tmp.replace(work_faiss)

    # Vérification
    with open(str(work_pkl), "rb") as f:
        vg = pickle.load(f)
    vi = faiss.read_index(str(work_faiss))
    n_emb = len(vg.get("label_to_idxs", {}).get(label, []))
    assert label in vg.get("products", []), f"'{label}' absent après sauvegarde"
    assert vi.ntotal == index.ntotal, "FAISS ntotal mismatch"

    log.info(f"  PKL sauvegardé ✓ ({work_pkl.stat().st_size/1024:.0f} KB)")
    log.info(f"  FAISS sauvegardé ✓ ({work_faiss.stat().st_size/1024:.0f} KB)")
    log.info(f"  ✅ '{label}' : {n_emb} emb, FAISS={vi.ntotal}")
    log.info(f"  → Galerie de travail : {work_pkl}")
    return len(new_embs)


def _remove_embeddings_from_gallery(label: str) -> dict:
    """Supprime tous les embeddings d'un label du PKL et reconstruit le FAISS."""
    import faiss, numpy as np

    _ensure_work_copies()
    work_pkl   = Path(GALLERY_PKL_WORK)
    work_faiss = Path(FAISS_INDEX_WORK)

    with open(str(work_pkl), "rb") as f:
        gdata = pickle.load(f)

    all_labels   = list(gdata.get("all_labels", gdata.get("labels", [])))
    label_to_idx = dict(gdata.get("label_to_idxs", {}))
    centroids    = dict(gdata.get("centroids", {}))
    label_counts = dict(gdata.get("label_counts", {}))
    products     = list(gdata.get("products", []))
    dim          = gdata.get("embed_dim", 1024)

    if label not in products and label not in label_to_idx:
        log.warning(f"  [Remove] '{label}' introuvable dans la galerie")
        return {"not_found": True}

    old_index  = faiss.read_index(str(work_faiss))
    to_remove  = set(label_to_idx.get(label, []))
    n_removed  = len(to_remove)

    if n_removed == 0:
        log.warning(f"  [Remove] '{label}' : 0 embeddings trouvés")
    else:
        keep_indices = [i for i in range(old_index.ntotal) if i not in to_remove]
        new_index    = faiss.IndexFlatIP(dim)

        if keep_indices:
            batch = np.zeros((len(keep_indices), dim), dtype="float32")
            for new_i, old_i in enumerate(keep_indices):
                batch[new_i] = old_index.reconstruct(old_i)
            new_index.add(batch)

        old_to_new = {old_i: new_i for new_i, old_i in enumerate(keep_indices)}
        new_label_to_idx = {}
        for lbl, idxs in label_to_idx.items():
            if lbl == label:
                continue
            new_idxs = [old_to_new[i] for i in idxs if i in old_to_new]
            if new_idxs:
                new_label_to_idx[lbl] = new_idxs

        new_all_labels = [l for l in all_labels if l != label]

        centroids.pop(label, None)
        label_counts.pop(label, None)
        if label in products:
            products.remove(label)

        gdata.update({
            "all_labels":    new_all_labels,
            "labels":        new_all_labels,
            "centroids":     centroids,
            "label_counts":  label_counts,
            "label_to_idxs": new_label_to_idx,
            "products":      sorted(set(products)),
            "n_raw_images":  new_index.ntotal,
        })

        pkl_tmp = work_pkl.with_suffix(".pkl.tmp")
        with open(str(pkl_tmp), "wb") as f:
            pickle.dump(gdata, f)
        pkl_tmp.replace(work_pkl)

        faiss_tmp = work_faiss.with_suffix(".index.tmp")
        faiss.write_index(new_index, str(faiss_tmp))
        faiss_tmp.replace(work_faiss)

        log.info(f"  [Remove] ✅ '{label}' supprimé : {n_removed} emb retirés, FAISS={new_index.ntotal}")

    return {"removed": n_removed, "label": label}


# ════════════════════════════════════════════════════════════════════
#  Kaggle API helpers
# ════════════════════════════════════════════════════════════════════
_kaggle_api = None
_kaggle_ok  = False

def get_kaggle_api():
    global _kaggle_api, _kaggle_ok
    if _kaggle_api is not None:
        return _kaggle_api if _kaggle_ok else None
    try:
        import kaggle
        kaggle.api.authenticate()
        _kaggle_api = kaggle.api; _kaggle_ok = True
        try:    username = kaggle.api.get_config_value("username")
        except: username = "unknown"
        log.info(f"Kaggle ✓ (user: {username})")
        return _kaggle_api
    except Exception:
        pass
    try:
        from kaggle.api.kaggle_api_extended import KaggleApiExtended
        api = KaggleApiExtended(); api.authenticate()
        _kaggle_api = api; _kaggle_ok = True
        log.info("Kaggle ✓ (legacy)")
        return _kaggle_api
    except Exception as e:
        log.warning(f"Kaggle non dispo : {e}"); _kaggle_ok = False; return None


def notify_server_reload():
    """Tell server-v1 to hot-reload its gallery after a push."""
    if not SERVER_V1_URL:
        log.info("[Notify] SERVER_V1_URL not set — skipping reload signal")
        return
    try:
        import urllib.request
        url = f"{SERVER_V1_URL.rstrip('/')}/api/reload-gallery"
        req = urllib.request.Request(
            url, data=b"{}",
            method="POST",
            headers={
                "Content-Type": "application/json",
                "X-Reload-Secret": SERVER_V1_RELOAD_SECRET
            }
        )
        with urllib.request.urlopen(req, timeout=10) as resp:
            log.info(f"[Notify] ✅ Server reload triggered ({resp.status})")
    except Exception as e:
        log.warning(f"[Notify] Could not reach server-v1: {e}")


# ════════════════════════════════════════════════════════════════════
#  Lock IMAGES et lock GALERIE séparés — plus de deadlock
# ════════════════════════════════════════════════════════════════════
_sync_lock    = threading.Lock()   # dataset images (sarahlaouedj25/...)
_gallery_lock = threading.Lock()   # dataset galerie (abdelghaniyacine/...)
_sync_status    = {"state": "idle",  "message": "", "last_sync": None}
_gallery_status = {"state": "idle",  "message": "", "last_push": None}

def _upd_sync(state, msg):
    _sync_status.update({"state": state, "message": msg,
                          "last_sync": datetime.now().isoformat()})
    log.info(f"[Images] {state} — {msg}")

def _upd_gallery(state, msg):
    _gallery_status.update({"state": state, "message": msg,
                             "last_push": datetime.now().isoformat()})
    log.info(f"[Gallery] {state} — {msg}")

def _update_status(state, msg):
    _upd_sync(state, msg)


# ════════════════════════════════════════════════════════════════════
#  Push galerie PKL+FAISS → abdelghaniyacine/gallary-dino-76
# ════════════════════════════════════════════════════════════════════
def push_gallery_to_kaggle(label: str = "", notes: str = ""):
    api = get_kaggle_api()
    if api is None:
        log.warning("  [Gallery push] Kaggle non dispo — galerie locale uniquement")
        return {"local_only": True}

    work_pkl   = Path(GALLERY_PKL_WORK)
    work_faiss = Path(FAISS_INDEX_WORK)
    if not work_pkl.exists() or not work_faiss.exists():
        return {"error": "Work copies not found"}

    note = notes or (f"Add '{label}' embeddings" if label else "Update gallery")

    with _gallery_lock:
        _upd_gallery("pushing", f"Push PKL+FAISS → {KAGGLE_MODEL_DATASET} ({note})")
        tmp_dir = None
        try:
            import zipfile
            tmp_dir  = Path(tempfile.mkdtemp(prefix="sb_gal_"))
            zip_path = tmp_dir / "gallery_update.zip"

            owner, ds_name = KAGGLE_MODEL_DATASET.split("/")
            meta = {"title": "SmartBasket DINOv2 Gallery",
                    "id": f"{owner}/{ds_name}",
                    "licenses": [{"name": "CC0-1.0"}]}
            meta_path = tmp_dir / "dataset-metadata.json"
            meta_path.write_text(json.dumps(meta, indent=2))

            with zipfile.ZipFile(str(zip_path), "w", zipfile.ZIP_DEFLATED) as zf:
                zf.write(str(meta_path),  "dataset-metadata.json")
                zf.write(str(work_pkl),   "gallery_finetuned_v3.pkl")
                zf.write(str(work_faiss), "gallery_finetuned_v3.index")
                if Path(FINETUNED_CHECKPOINT).exists():
                    zf.write(FINETUNED_CHECKPOINT, "dinov2_finetuned_supcon_v3.pt")

            sz_mb = zip_path.stat().st_size / 1024 / 1024

            with open(str(work_pkl), "rb") as f:
                gdata = pickle.load(f)
            n_prods = len(gdata.get("products", []))
            n_embs  = gdata.get("n_raw_images", 0)

            log.info(f"  [Gallery push] Zip : {sz_mb:.1f} MB | {n_prods} produits | {n_embs} emb")
            log.info(f"  [Gallery push] Upload vers {KAGGLE_MODEL_DATASET}…")

            try:
                api.dataset_create_version_by_path(
                    path=str(zip_path), version_notes=note,
                    quiet=False, convert_to_csv=False, delete_old_versions=False)
                method = "by_path"
            except (AttributeError, Exception) as _e1:
                log.warning(f"  [Gallery push] by_path échoué ({_e1}) → fallback dossier")
                push_dir = tmp_dir / "gallery_push_dir"
                push_dir.mkdir(exist_ok=True)
                (push_dir / "dataset-metadata.json").write_text(
                    json.dumps({"title": "SmartBasket DINOv2 Gallery",
                                "id": f"{KAGGLE_MODEL_DATASET}",
                                "licenses": [{"name": "CC0-1.0"}]}, indent=2))
                shutil.copy2(str(work_pkl),   str(push_dir / "gallery_finetuned_v3.pkl"))
                shutil.copy2(str(work_faiss), str(push_dir / "gallery_finetuned_v3.index"))
                if Path(FINETUNED_CHECKPOINT).exists():
                    shutil.copy2(FINETUNED_CHECKPOINT,
                                 str(push_dir / "dinov2_finetuned_supcon_v3.pt"))
                api.dataset_create_version(
                    folder=str(push_dir), version_notes=note,
                    quiet=False, dir_mode="zip", delete_old_versions=False)
                method = "dir_mode"

            _upd_gallery("idle",
                f"✅ Galerie poussée ({n_prods} produits, {n_embs} emb) [{method}]")
            log.info(f"  [Gallery push] ✅ Succès — {n_prods} produits, {n_embs} emb")
            threading.Thread(target=notify_server_reload, daemon=True).start()
            return {"pushed": True, "n_products": n_prods, "n_embeddings": n_embs, "method": method}

        except Exception as e:
            _upd_gallery("error", str(e))
            log.error(f"  [Gallery push] ECHEC : {e}")
            return {"error": str(e)}
        finally:
            if tmp_dir and Path(str(tmp_dir)).exists():
                shutil.rmtree(str(tmp_dir), ignore_errors=True)


# ════════════════════════════════════════════════════════════════════
#  Pipeline complet : YOLO → DINOv2 → push images → push galerie
# ════════════════════════════════════════════════════════════════════
def process_product_images(label: str, image_paths: list,
                           is_update: bool = False, old_label: str = None):
    log.info(f"[Process] '{label}' — {len(image_paths)} image(s)")
    try:
        _load_models()
    except Exception as e:
        log.error(f"[Process] Modèles : {e}"); return

    total_dets = 0
    for img_path in image_paths:
        try:
            n = _run_yolo_on_image(img_path, label)
            total_dets += n
            log.info(f"  YOLO {img_path.name} → {n} dét.")
        except Exception as e:
            log.warning(f"  YOLO {img_path.name} : {e}")

    crop_dir  = prod_root(label) / "croppedimages"
    crop_imgs = ([p for p in sorted(crop_dir.iterdir())
                  if p.is_file() and p.suffix.lower() in ALLOWED_EXTS]
                 if crop_dir.exists() else [])
    embed_src = crop_imgs if crop_imgs else list(image_paths)
    log.info(f"  Embed src : {len(embed_src)} {'crop(s)' if crop_imgs else 'image(s) originale(s)'}")

    try:
        n_emb = _add_embeddings_to_gallery(label, embed_src)
    except Exception as e:
        log.error(f"[Process] Embeddings : {e}")
        log.error("[Process] Push annulé (embeddings non écrits)")
        return

    if n_emb == 0:
        log.warning("[Process] 0 embeddings → push annulé")
        return

    log.info(f"[Process] ✓ {total_dets} dét. YOLO, {n_emb} emb")

    # Push images ET galerie en parallèle (locks différents)
    def _push_images():
        try:
            if is_update:
                note = (f"Rename '{old_label}'→'{label}' + images"
                        if old_label and old_label != label
                        else f"Update: {label}")
                r = kaggle_push_gallery(notes=note)
            else:
                r = kaggle_push_gallery(notes=f"Add product: {label}")
            log.info(f"[Process] Dataset images push : {r}")
        except Exception as e:
            log.error(f"[Process] Dataset images push : {e}")

    def _push_gallery():
        note = (f"Update '{label}' — {n_emb} emb (update)"
                if is_update else f"Add '{label}' — {n_emb} emb")
        r = push_gallery_to_kaggle(label=label, notes=note)
        log.info(f"[Process] Gallery push : {r}")

    t_imgs    = threading.Thread(target=_push_images, daemon=True)
    t_gallery = threading.Thread(target=_push_gallery, daemon=True)
    t_imgs.start()
    t_gallery.start()


# ════════════════════════════════════════════════════════════════════
#  CORS
# ════════════════════════════════════════════════════════════════════
@app.before_request
def handle_preflight():
    if request.method == "OPTIONS":
        r = Response()
        r.headers.update({"Access-Control-Allow-Origin": "*",
                          "Access-Control-Allow-Methods": "GET,POST,PUT,DELETE,OPTIONS",
                          "Access-Control-Allow-Headers": "Content-Type,ngrok-skip-browser-warning,Authorization"})
        return r

@app.after_request
def add_cors_headers(resp):
    resp.headers.update({"Access-Control-Allow-Origin": "*",
                         "Access-Control-Allow-Methods": "GET,POST,PUT,DELETE,OPTIONS",
                         "Access-Control-Allow-Headers": "Content-Type,ngrok-skip-browser-warning,Authorization"})
    return resp


# ════════════════════════════════════════════════════════════════════
#  Kaggle dataset IMAGES (sarahlaouedj25/...)
# ════════════════════════════════════════════════════════════════════
def _ensure_dataset_metadata():
    meta = gallery() / "dataset-metadata.json"
    if not meta.exists():
        owner, ds = KAGGLE_IMAGES_DATASET.split("/")
        meta.write_text(json.dumps({"title":"Smartbasket Dataset Multiside",
                                    "id":f"{owner}/{ds}","licenses":[{"name":"CC0-1.0"}]}, indent=2))
    return meta

def _find_gallery_root(base: Path):
    subdirs = [d for d in base.iterdir() if d.is_dir()]
    if not subdirs: return None
    if len(subdirs) == 1:
        inner = [d for d in subdirs[0].iterdir() if d.is_dir()]
        if inner: return subdirs[0]
    return base

def kaggle_pull_gallery(force=False):
    api = get_kaggle_api()
    if api is None: return {"error": "Kaggle not authenticated"}
    gallery().mkdir(parents=True, exist_ok=True)
    if any(p for p in gallery().iterdir() if p.is_dir()) and not force:
        return {"skipped": True, "products": len(list_products())}
    with _sync_lock:
        _upd_sync("pulling", f"Downloading {KAGGLE_IMAGES_DATASET}…")
        try:
            with tempfile.TemporaryDirectory() as tmp:
                api.dataset_download_files(KAGGLE_IMAGES_DATASET, path=tmp,
                                           unzip=True, quiet=False, force=True)
                root = _find_gallery_root(Path(tmp))
                if root is None: return {"error": "gallery root not found"}
                n = 0
                for d in sorted(root.iterdir()):
                    if not d.is_dir(): continue
                    dest = gallery() / d.name
                    if dest.exists(): shutil.rmtree(str(dest))
                    shutil.copytree(str(d), str(dest)); n += 1
            _upd_sync("idle", f"Pulled {n} products")
            return {"synced": n}
        except Exception as e:
            _upd_sync("error", str(e)); return {"error": str(e)}

def kaggle_push_gallery(notes=PUSH_NOTES):
    """
    ═══════════════════════════════════════════════════════
    FIX v5-fixed4 : pousse TOUTE la galerie locale vers
    Kaggle (tous les produits), pas seulement un seul.
    ═══════════════════════════════════════════════════════
    """
    api = get_kaggle_api()
    if api is None:
        return {"local_only": True}

    with _sync_lock:
        _upd_sync("pushing", "Full gallery push…")
        tmp_dir = None
        try:
            _ensure_dataset_metadata()

            tmp_dir  = Path(tempfile.mkdtemp(prefix="sb_fullpush_"))
            push_dir = tmp_dir / "push_content"
            push_dir.mkdir(parents=True, exist_ok=True)

            # ── dataset-metadata.json ──────────────────────────────
            owner, ds = KAGGLE_IMAGES_DATASET.split("/")
            meta = {
                "title": "Smartbasket Dataset Multiside",
                "id": f"{owner}/{ds}",
                "licenses": [{"name": "CC0-1.0"}]
            }
            (push_dir / "dataset-metadata.json").write_text(json.dumps(meta, indent=2))

            # ── Copier TOUS les dossiers produits ──────────────────
            n_copied = 0
            for item in sorted(gallery().iterdir()):
                if not item.is_dir():
                    continue
                dest = push_dir / item.name
                shutil.copytree(str(item), str(dest), dirs_exist_ok=True)
                n_copied += 1
                log.info(f"  → Copié : {item.name}")

            log.info(f"  [Full push] {n_copied} produit(s) dans push_dir")

            api.dataset_create_version(
                folder=str(push_dir),
                version_notes=notes,
                quiet=False,
                dir_mode="zip",
                delete_old_versions=False
            )

            _upd_sync("idle", f"✅ Full push ✓ ({n_copied} produits)")
            return {"pushed": True, "n_products": n_copied}

        except Exception as e:
            _upd_sync("error", str(e))
            log.error(f"[Full push] ECHEC : {e}")
            return {"error": str(e)}
        finally:
            if tmp_dir and tmp_dir.exists():
                shutil.rmtree(str(tmp_dir), ignore_errors=True)


def kaggle_push_product(label: str, notes: str = ""):
    """
    ═══════════════════════════════════════════════════════
    FIX v5-fixed4 : copie TOUTE la galerie locale dans
    push_dir (pas seulement le produit concerné) pour que
    Kaggle reçoive le dataset complet à chaque version.
    ═══════════════════════════════════════════════════════
    """
    api = get_kaggle_api()
    if api is None:
        return {"local_only": True}

    with _sync_lock:
        _upd_sync("pushing", f"Push complet (produit '{label}' inclus) → nouvelle version")
        tmp_dir = None
        try:
            _ensure_dataset_metadata()

            tmp_dir  = Path(tempfile.mkdtemp(prefix="sb_push_"))
            push_dir = tmp_dir / "push_content"
            push_dir.mkdir(parents=True, exist_ok=True)

            # ── dataset-metadata.json ──────────────────────────────
            owner, ds = KAGGLE_IMAGES_DATASET.split("/")
            meta = {
                "title": "Smartbasket Dataset Multiside",
                "id": f"{owner}/{ds}",
                "licenses": [{"name": "CC0-1.0"}]
            }
            (push_dir / "dataset-metadata.json").write_text(json.dumps(meta, indent=2))

            # ── Copier TOUS les produits de la galerie locale ──────
            # (le produit `label` y est déjà car ensure_structure() a été appelé avant)
            n_copied = 0
            for item in sorted(gallery().iterdir()):
                if not item.is_dir():
                    continue
                dest = push_dir / item.name
                shutil.copytree(str(item), str(dest), dirs_exist_ok=True)
                n_copied += 1
                log.info(f"  → Copié : {item.name}")

            log.info(f"  [Push product] {n_copied} produit(s) dans push_dir "
                     f"(produit cible : '{safe_label(label)}')")

            note = notes or f"Add/Update: {label}"

            api.dataset_create_version(
                folder=str(push_dir),
                version_notes=note,
                quiet=False,
                dir_mode="zip",
                delete_old_versions=False
            )

            _upd_sync("idle", f"✅ '{label}' poussé — dataset complet ({n_copied} produits)")
            return {"pushed": True, "product": safe_label(label),
                    "n_products_total": n_copied, "version": "new"}

        except Exception as e:
            _upd_sync("error", str(e))
            log.error(f"Push product failed: {e}")
            return {"error": str(e)}
        finally:
            if tmp_dir and tmp_dir.exists():
                shutil.rmtree(str(tmp_dir), ignore_errors=True)


def kaggle_delete_product(label):
    # Suppression = push galerie complète sans le produit (déjà retiré localement)
    return kaggle_push_gallery(notes=f"Delete: {label}")

def kaggle_delete_image(label, filename):
    return kaggle_push_product(label, notes=f"Del image {filename}")

def push_async(fn, *args):
    threading.Thread(target=fn, args=args, daemon=True).start()


# ════════════════════════════════════════════════════════════════════
#  Helpers galerie locale
# ════════════════════════════════════════════════════════════════════
def gallery():   return LOCAL_CACHE_DIR
def safe_label(r): return re.sub(r"[^\w\s\-\.]","",r).strip().replace(" ","_")
def label_to_display(f): return f.replace("_"," ")
def prod_root(l): return gallery() / safe_label(l)
def images_d(l):  return prod_root(l) / "images"
def cropped_d(l): return prod_root(l) / "croppedimages"

def ensure_structure(l):
    for d in [prod_root(l), images_d(l), cropped_d(l), prod_root(l)/"labels"]:
        d.mkdir(parents=True, exist_ok=True)

def count_images(l):
    for d in [images_d(l), cropped_d(l), prod_root(l)]:
        if d.exists():
            n = sum(1 for f in d.iterdir() if f.is_file() and f.suffix.lower() in ALLOWED_EXTS)
            if n: return n
    return 0

def get_thumb_b64(l):
    for d in [images_d(l), cropped_d(l), prod_root(l)]:
        if not d.exists(): continue
        for f in sorted(d.iterdir()):
            if f.is_file() and f.suffix.lower() in ALLOWED_EXTS:
                try:
                    img = Image.open(f).convert("RGB"); img.thumbnail(THUMB_SIZE, Image.LANCZOS)
                    buf = io.BytesIO(); img.save(buf,"JPEG",quality=75)
                    return base64.b64encode(buf.getvalue()).decode()
                except: pass
    return None

def stat_mtime(l):
    d = prod_root(l)
    return datetime.fromtimestamp(d.stat().st_mtime).strftime("%b %d, %Y") if d.exists() else ""

def list_products():
    if not gallery().exists(): return []
    prods = []
    for i, p in enumerate(sorted(gallery().iterdir()), 1):
        if not p.is_dir() or p.name.startswith("."): continue
        l = label_to_display(p.name); n = count_images(l)
        prods.append({"id":i,"folder":p.name,"label":l,"images":n,
                      "date":stat_mtime(l),"status":"active" if n>=1 else "draft"})
    return prods

def get_image_list(l):
    imgs = []
    for sd in ["images","croppedimages"]:
        d = prod_root(l)/sd
        if d.exists():
            for f in sorted(d.iterdir()):
                if f.is_file() and f.suffix.lower() in ALLOWED_EXTS:
                    imgs.append({"dir":sd,"name":f.name})
    for f in sorted(prod_root(l).iterdir()):
        if f.is_file() and f.suffix.lower() in ALLOWED_EXTS:
            imgs.append({"dir":"__root__","name":f.name})
    return imgs


# ════════════════════════════════════════════════════════════════════
#  API ROUTES
# ════════════════════════════════════════════════════════════════════

@app.get("/api/kaggle/status")
def api_kaggle_status():
    api = get_kaggle_api()
    username = None
    if api:
        try: username = api.get_config_value("username")
        except: pass
    work_info = {}
    try:
        with open(_gallery_pkl_path(),"rb") as f: g = pickle.load(f)
        work_info = {"n_products": len(g.get("products",[])),
                     "n_embeddings": g.get("n_raw_images",0),
                     "path": _gallery_pkl_path(),
                     "is_work_copy": Path(_gallery_pkl_path())==Path(GALLERY_PKL_WORK)}
    except: pass
    return jsonify({"authenticated":_kaggle_ok,"username":username,
                    "images_dataset":KAGGLE_IMAGES_DATASET,"model_dataset":KAGGLE_MODEL_DATASET,
                    "sync":_sync_status,"gallery_sync":_gallery_status,
                    "local_cache":str(gallery()),"work_dir":str(_WORK_DIR),
                    "gallery_pkl":_gallery_pkl_path(),"work_gallery":work_info})

@app.post("/api/kaggle/pull")
def api_kaggle_pull():
    return jsonify(kaggle_pull_gallery((request.json or {}).get("force",False)))

@app.post("/api/kaggle/push")
def api_kaggle_push():
    return jsonify(kaggle_push_gallery((request.json or {}).get("notes",PUSH_NOTES)))

@app.post("/api/gallery/push_work_copies")
def api_push_work_copies():
    notes  = (request.json or {}).get("notes","Manual gallery push")
    result = push_gallery_to_kaggle(notes=notes)
    return jsonify(result)

@app.get("/api/gallery/stats")
def api_gallery_stats():
    prods = list_products()
    gi = {}
    try:
        with open(_gallery_pkl_path(),"rb") as f: g = pickle.load(f)
        gi = {"n_products":len(g.get("products",[])),"n_embeddings":g.get("n_raw_images",0),
              "embed_dim":g.get("embed_dim","?"),"gallery_pkl":_gallery_pkl_path(),
              "is_work_copy":Path(_gallery_pkl_path())==Path(GALLERY_PKL_WORK),
              "gallery_status":_gallery_status}
    except: pass
    return jsonify({"total_products":len(prods),"total_images":sum(p["images"] for p in prods),
                    "gallery_path":str(gallery()),"dino_gallery":gi})

@app.get("/api/products")
def api_list_products():
    q = request.args.get("q","").lower()
    prods = list_products()
    if q: prods = [p for p in prods if q in p["label"].lower() or q in p["folder"].lower()]
    return jsonify({"products":prods,"total":len(prods)})

@app.get("/api/products/<folder>")
def api_get_product(folder):
    l = label_to_display(folder); r = prod_root(l)
    if not r.exists(): return jsonify({"error":"Not found"}),404
    ld = r/"labels"
    return jsonify({"folder":folder,"label":l,"images":count_images(l),
                    "status":"active" if count_images(l)>=1 else "draft",
                    "date":stat_mtime(l),"image_list":get_image_list(l),
                    "label_files":sum(1 for f in ld.iterdir() if f.suffix==".txt") if ld.exists() else 0,
                    "paths":{"root":str(r),"images":str(images_d(l)),
                             "cropped":str(cropped_d(l)),"labels":str(ld)}})

@app.get("/api/products/<folder>/thumbnail")
def api_thumbnail(folder):
    t = get_thumb_b64(label_to_display(folder))
    return jsonify({"thumbnail":t})

@app.get("/api/products/<folder>/gallery_check")
def api_gallery_check(folder):
    l = label_to_display(folder)
    res = {"folder":folder,"label":l,"in_gallery":False,"n_embeddings":0,
           "faiss_ntotal":0,"gallery_pkl":_gallery_pkl_path(),
           "is_work_copy":Path(_gallery_pkl_path())==Path(GALLERY_PKL_WORK),
           "gallery_status":_gallery_status,"error":None,
           "work_dir":str(_WORK_DIR),"work_pkl":GALLERY_PKL_WORK,"work_faiss":FAISS_INDEX_WORK}
    try:
        import faiss
        with open(_gallery_pkl_path(),"rb") as f: g = pickle.load(f)
        idx = faiss.read_index(_faiss_index_path())
        res.update({"in_gallery":l in g.get("products",[]),
                    "n_embeddings":len(g.get("label_to_idxs",{}).get(l,[])),
                    "faiss_ntotal":idx.ntotal,"dim_ok":idx.d==g.get("embed_dim",1024),
                    "n_products_total":len(g.get("products",[]))})
    except Exception as e:
        res["error"] = str(e)
    return jsonify(res)



# ── Price service ─────────────────────────────────────────────────────────────
import csv as _csv, random as _random

_LOCAL_PRICES_PATH   = Path.home() / ".smartbasket" / "products_prices.csv"
_KAGGLE_PRICES_INPUT = "/kaggle/input/datasets/sarahlaouedj25/product-prices/products_prices.csv"
_prices_lock = threading.Lock()

def _load_local_prices():
    if not _LOCAL_PRICES_PATH.exists():
        return {}
    try:
        db = {}
        with open(_LOCAL_PRICES_PATH, newline="", encoding="utf-8") as _f:
            for row in _csv.DictReader(_f):
                lbl = (row.get("label") or "").strip()
                try:
                    prc = int(float((row.get("price") or "0").strip()))
                except (ValueError, AttributeError):
                    continue
                if lbl:
                    db[lbl] = prc
        return db
    except Exception as _e:
        log.warning(f"Price load error: {_e}")
        return {}

def _save_local_prices(db):
    try:
        _LOCAL_PRICES_PATH.parent.mkdir(parents=True, exist_ok=True)
        with open(_LOCAL_PRICES_PATH, "w", newline="", encoding="utf-8") as _f:
            w = _csv.writer(_f)
            w.writerow(["label", "price"])
            for lbl, prc in sorted(db.items()):
                w.writerow([lbl, prc])
    except Exception as _e:
        log.warning(f"Price save error: {_e}")

def _bootstrap_prices_from_kaggle():
    """Seed local prices CSV from Kaggle input dataset on first run."""
    if _LOCAL_PRICES_PATH.exists():
        return
    if not Path(_KAGGLE_PRICES_INPUT).exists():
        return
    try:
        import shutil as _sh
        _LOCAL_PRICES_PATH.parent.mkdir(parents=True, exist_ok=True)
        _sh.copy2(_KAGGLE_PRICES_INPUT, str(_LOCAL_PRICES_PATH))
        log.info(f"Prices bootstrapped from Kaggle input → {_LOCAL_PRICES_PATH}")
    except Exception as _e:
        log.warning(f"Price bootstrap failed: {_e}")

_bootstrap_prices_from_kaggle()

def _estimate_price(label: str) -> int:
    l = label.lower()
    if any(k in l for k in ["oil", "huile", "recamar", "olive", "tournesol"]):
        return _random.randint(280, 480)
    if any(k in l for k in ["water", "eau", "atlas", "0.5l", "1.5l", "minera"]):
        return _random.randint(20, 65)
    if any(k in l for k in ["milk", "lait", "smen", "margarin", "beurre", "fromage"]):
        return _random.randint(120, 400)
    if any(k in l for k in ["juice", "jus", "rouiba", "orage", "agrume", "cootal"]):
        return _random.randint(50, 160)
    if any(k in l for k in ["choco", "biscuit", "cracker", "gateau", "noisette", "candy"]):
        return _random.randint(50, 220)
    if any(k in l for k in ["gel", "shampoo", "savon", "chlorox", "deterg", "clean", "venus"]):
        return _random.randint(100, 680)
    if any(k in l for k in ["pasta", "pate", "spaghetti", "macar", "mahbouba"]):
        return _random.randint(80, 180)
    if any(k in l for k in ["the ", "thé", "tea", "vitl", "chaina", "vert"]):
        return _random.randint(200, 320)
    if any(k in l for k in ["ramdy", "fromage", "kiri", "vache"]):
        return _random.randint(200, 450)
    return _random.randint(80, 350)

def upsert_price_local(label: str, price: int):
    with _prices_lock:
        db = _load_local_prices()
        db[label] = price
        _save_local_prices(db)

def _notify_server_price_upsert(label: str, price: int):
    try:
        import requests
        requests.post(
            f"{SERVER_V1_URL}/api/price-upsert",
            json={"label": label, "price": price},
            timeout=5,
        )
    except Exception as _e:
        log.warning(f"Price upsert to server failed: {_e}")

def remove_price_local(label: str):
    with _prices_lock:
        db = _load_local_prices()
        db.pop(label, None)
        _save_local_prices(db)

def _notify_server_price_remove(label: str):
    try:
        import requests
        requests.delete(f"{SERVER_V1_URL}/api/price-remove/{label}", timeout=5)
    except Exception as _e:
        log.warning(f"Price remove from server failed: {_e}")

@app.post("/api/products")
def api_add_product():
    label = (request.form.get("label") or "").strip()
    if not label:
        return jsonify({"error": "label required"}), 400

    folder = safe_label(label)
    if (gallery() / folder).exists():
        return jsonify({"error": f"'{label}' existe déjà"}), 409

    # Crée : gallery/<folder>/images/  croppedimages/  labels/
    ensure_structure(label)

    # Assign price from CSV price service
    _new_price = _estimate_price(label)
    upsert_price_local(label, _new_price)
    threading.Thread(target=_notify_server_price_upsert, args=(label, _new_price), daemon=True).start()

    files = request.files.getlist("images")
    saved = []
    for i, f in enumerate(files):
        if not f or not f.filename:
            continue
        ext = Path(f.filename).suffix.lower()
        if ext not in ALLOWED_EXTS:
            continue
        dest = images_d(label) / f"{folder}_{i:04d}{ext}"
        f.save(str(dest))
        saved.append(dest)

    if saved:
        # YOLO + DINOv2 + push images (galerie complète) + push galerie PKL/FAISS
        threading.Thread(
            target=process_product_images,
            args=(label, saved),
            daemon=True
        ).start()
    else:
        # Pas d'images : push galerie complète quand même
        push_async(kaggle_push_gallery, f"Create empty product: {label}")

    return jsonify({
        "folder": folder,
        "label": label,
        "images": len(saved),
        "message": "Produit créé. Push dataset complet vers Kaggle lancé."
    }), 201


@app.put("/api/products/<folder>")
def api_update_product(folder):
    label     = label_to_display(folder)
    new_label = (request.form.get("label") or "").strip()
    root      = prod_root(label)
    if not root.exists(): return jsonify({"error":"Not found"}),404
    old_lbl = label
    if new_label and safe_label(new_label) != folder:
        new_folder = safe_label(new_label)
        if (gallery()/new_folder).exists(): return jsonify({"error":f"'{new_label}' exists"}),409
        shutil.move(str(root), str(gallery()/new_folder))
        label = new_label; folder = new_folder
    ensure_structure(label)
    files = request.files.getlist("images")
    saved = []
    ec = count_images(label)
    for i, f in enumerate(files):
        if not f or not f.filename: continue
        ext = Path(f.filename).suffix.lower()
        if ext not in ALLOWED_EXTS: continue
        dest = images_d(label) / f"{safe_label(label)}_{ec+i:04d}{ext}"
        f.save(str(dest)); saved.append(dest)
    if saved:
        threading.Thread(target=process_product_images,
                         args=(label,saved), kwargs={"is_update":True,"old_label":old_lbl},
                         daemon=True).start()
    else:
        note = (f"Rename→'{label}'" if old_lbl != label else f"Update: {label}")
        push_async(kaggle_push_gallery, note)
        push_async(push_gallery_to_kaggle, label, note)
    return jsonify({"folder":folder,"label":label,"images":count_images(label),
                    "kaggle_push":"queued" if _kaggle_ok else "local_only",
                    "check_url":f"/api/products/{folder}/gallery_check"})

@app.delete("/api/products/<folder>")
def api_delete_product(folder):
    l = label_to_display(folder)
    r = prod_root(l)

    if not r.exists():
        return jsonify({"error": "Not found"}), 404

    shutil.rmtree(str(r))
    log.info(f"'{l}' supprimé localement ✓")

    removal = _remove_embeddings_from_gallery(l)

    # Remove from price service
    remove_price_local(l)
    threading.Thread(target=_notify_server_price_remove, args=(l,), daemon=True).start()

    # Push galerie complète (sans le produit supprimé)
    push_async(kaggle_push_gallery, f"Delete product: {l}")

    threading.Thread(
        target=push_gallery_to_kaggle,
        args=(l,),
        kwargs={"notes": f"Delete '{l}' — embeddings removed"},
        daemon=True
    ).start()

    return jsonify({
        "deleted": folder,
        "embeddings": removal,
        "kaggle_push": "queued"
    })


@app.get("/api/products/<folder>/image/<subdir>/<filename>")
def api_serve_image(folder, subdir, filename):
    l = label_to_display(folder); r = prod_root(l)
    p = r/filename if subdir=="__root__" else r/subdir/filename
    if not p.exists(): return jsonify({"error":"Not found"}),404
    try:
        data = base64.b64encode(open(p,"rb").read()).decode()
        ext  = p.suffix.lower().lstrip(".")
        mime = {"jpg":"jpeg","jpeg":"jpeg","png":"png","webp":"webp","bmp":"bmp"}.get(ext,"jpeg")
        return jsonify({"data":f"data:image/{mime};base64,{data}"})
    except Exception as e:
        return jsonify({"error":str(e)}),500

@app.delete("/api/products/<folder>/image/<subdir>/<filename>")
def api_delete_image(folder, subdir, filename):
    l = label_to_display(folder); r = prod_root(l)
    p = r/filename if subdir=="__root__" else r/subdir/filename
    if not p.exists(): return jsonify({"error":"Not found"}),404
    p.unlink(); push_async(kaggle_delete_image, l, filename)
    return jsonify({"deleted":filename,"kaggle_push":"queued" if _kaggle_ok else "local_only"})

@app.get("/api/products/<folder>/processing_status")
def api_processing_status(folder):
    l = label_to_display(folder); r = prod_root(l)
    ld = r/"labels"; cd = r/"croppedimages"; id_ = r/"images"
    ni = sum(1 for f in id_.iterdir() if f.is_file() and f.suffix.lower() in ALLOWED_EXTS) if id_.exists() else 0
    nl = sum(1 for f in ld.iterdir() if f.suffix==".txt") if ld.exists() else 0
    nc = sum(1 for f in cd.iterdir() if f.is_file() and f.suffix.lower() in ALLOWED_EXTS) if cd.exists() else 0
    ig,ne = False,0
    try:
        with open(_gallery_pkl_path(),"rb") as f: g = pickle.load(f)
        ig = l in g.get("products",[]); ne = len(g.get("label_to_idxs",{}).get(l,[]))
    except: pass
    return jsonify({"folder":folder,"label":l,"n_images":ni,"n_label_files":nl,"n_crops":nc,
                    "in_dino_gallery":ig,"n_embeddings":ne,"processing_done":(nl>=ni and ig),
                    "gallery_push_status":_gallery_status,"work_dir":str(_WORK_DIR)})


# ════════════════════════════════════════════════════════════════════
#  DÉMARRAGE
# ════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    log.info("="*70)
    log.info("🚀  SMARTBASKET ADMIN BACKEND  (v5-fixed4)")
    log.info(f"   Images  : {KAGGLE_IMAGES_DATASET}")
    log.info(f"   Gallery : {KAGGLE_MODEL_DATASET}")
    log.info(f"   Work    : {_WORK_DIR}")
    log.info(f"   PKL     : {_gallery_pkl_path()}")
    log.info("="*70)

    log.info("🔧 Init copies de travail…")
    try:
        _ensure_work_copies()
        log.info("✅ Copies de travail prêtes")
    except Exception as e:
        log.error(f"⚠️  Init échouée : {e}")

    get_kaggle_api()
    gallery().mkdir(parents=True, exist_ok=True)
    if not any(p for p in gallery().iterdir() if p.is_dir()):
        log.info("📥 Galerie vide → pull Kaggle…")
        threading.Thread(target=kaggle_pull_gallery, daemon=True).start()

    try:
        from pyngrok import ngrok
        ngrok.set_auth_token("3EEUbyUr1Nex1tJNTC72S5dYBMI_7GHqXyfhhWrVJzFc7eoJu")
        public_url = ngrok.connect(5000)
        print(f"\n🌐  URL publique : {public_url}\n")
    except Exception as e:
        log.warning(f"ngrok : {e}")

    port = int(os.environ.get("PORT", 5000))
    app.run(host="0.0.0.0", port=port, debug=False, use_reloader=False)

2026-05-31 15:48:06,746 | INFO | ======================================================================
2026-05-31 15:48:06,747 | INFO | 🚀  SMARTBASKET ADMIN BACKEND  (v5-fixed4)
2026-05-31 15:48:06,747 | INFO |    Images  : sarahlaouedj25/smartbasket-dataset-multiside
2026-05-31 15:48:06,748 | INFO |    Gallery : abdelghaniyacine/gallary-dino-76
2026-05-31 15:48:06,749 | INFO |    Work    : /kaggle/working/smartbasket_gallery
2026-05-31 15:48:06,749 | INFO |    PKL     : /kaggle/input/datasets/abdelghaniyacine/gallary-dino-76/gallery_finetuned_v3.pkl
2026-05-31 15:48:06,750 | INFO | ======================================================================
2026-05-31 15:48:06,750 | INFO | 🔧 Init copies de travail…


✅ Imports OK
✅ Work dir  : /kaggle/working/smartbasket_gallery
✅ PKL work  : /kaggle/working/smartbasket_gallery/gallery_finetuned_v3.pkl
✅ FAISS work: /kaggle/working/smartbasket_gallery/gallery_finetuned_v3.index


2026-05-31 15:48:06,780 | INFO |   PKL copié ✓ (1.2 MB)
2026-05-31 15:48:07,634 | INFO |   FAISS copié ✓ (43.4 MB)
2026-05-31 15:48:07,635 | INFO |   Work dir : /kaggle/working/smartbasket_gallery
2026-05-31 15:48:07,635 | INFO |     PKL   : /kaggle/working/smartbasket_gallery/gallery_finetuned_v3.pkl (1.2 MB)
2026-05-31 15:48:07,636 | INFO |     FAISS : /kaggle/working/smartbasket_gallery/gallery_finetuned_v3.index (43.4 MB)
2026-05-31 15:48:07,637 | INFO | ✅ Copies de travail prêtes
2026-05-31 15:48:08,403 | INFO | Kaggle ✓ (user: sarahlaouedj25)
2026-05-31 15:48:08,404 | INFO | 📥 Galerie vide → pull Kaggle…
2026-05-31 15:48:08,405 | INFO | [Images] pulling — Downloading sarahlaouedj25/smartbasket-dataset-multiside…


Dataset URL: https://www.kaggle.com/datasets/sarahlaouedj25/smartbasket-dataset-multiside


  0%|          | 0.00/465M [00:00<?, ?B/s]

...

  9%|▉         | 43.0M/465M [00:00<00:03, 136MB/s]

2026-05-31 15:48:09,302 | INFO | Updating authtoken for default "config_path" of "ngrok_path": /root/.config/ngrok/ngrok
 12%|█▏        | 57.0M/465M [00:00<00:04, 94.5MB/s]2026-05-31 15:48:09,348 | INFO | Opening tunnel named: http-5000-6a3bec7c-25fd-4889-aece-b796a69fa7ca
2026-05-31 15:48:09,369 | INFO | t=2026-05-31T15:48:09+0000 lvl=info msg="no configuration paths supplied"
2026-05-31 15:48:09,370 | INFO | t=2026-05-31T15:48:09+0000 lvl=info msg="using configuration at default config path" path=/root/.config/ngrok/ngrok.yml
2026-05-31 15:48:09,372 | INFO | t=2026-05-31T15:48:09+0000 lvl=info msg="open config file" path=/root/.config/ngrok/ngrok.yml err=nil
 15%|█▌        | 72.0M/465M [00:00<00:03, 109MB/s] 2026-05-31 15:48:09,499 | INFO | t=2026-05-31T15:48:09+0000 lvl=info msg="FIPS 140 mode" enabled=false
2026-05-31 15:48:09,505 | INFO | t=2026-05-31T15:48:09+0000 lvl=info msg="starting web service" obj=web addr=127.0.0.1:4040 allow_hosts=[]
 18%|█▊        | 84.0M/465M [00:00<00:


🌐  URL publique : NgrokTunnel: "https://prominent-purebred-hatchback.ngrok-free.dev" -> "http://localhost:5000"

 * Serving Flask app '__main__'
 * Debug mode: off


2026-05-31 15:48:10,247 | INFO | WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.19.2.2:5000
2026-05-31 15:48:10,248 | INFO | Press CTRL+C to quit
100%|██████████| 465M/465M [00:03<00:00, 123MB/s] 


2026-05-31 15:48:18,125 | INFO | [Images] idle — Pulled 302 products
2026-05-31 15:48:25,389 | INFO | t=2026-05-31T15:48:25+0000 lvl=info msg="join connections" obj=join id=8479c6e71432 l=127.0.0.1:5000 r=154.121.18.45:5375
2026-05-31 15:48:25,391 | INFO | 127.0.0.1 - - [31/May/2026 15:48:25] "OPTIONS /api/kaggle/status HTTP/1.1" 200 -
2026-05-31 15:48:25,776 | INFO | t=2026-05-31T15:48:25+0000 lvl=info msg="join connections" obj=join id=fb6c6243b3e4 l=127.0.0.1:5000 r=154.121.18.45:5375
2026-05-31 15:48:25,780 | INFO | 127.0.0.1 - - [31/May/2026 15:48:25] "GET /api/kaggle/status HTTP/1.1" 200 -
2026-05-31 15:48:40,027 | INFO | t=2026-05-31T15:48:40+0000 lvl=info msg="join connections" obj=join id=5237b5e3e291 l=127.0.0.1:5000 r=154.121.18.45:5375
2026-05-31 15:48:40,028 | INFO | 127.0.0.1 - - [31/May/2026 15:48:40] "OPTIONS /api/kaggle/status HTTP/1.1" 200 -
2026-05-31 15:48:40,344 | INFO | t=2026-05-31T15:48:40+0000 lvl=info msg="join connections" obj=join id=5f616475f2bb l=127.0.0.1